# Bulk scoring through the Team 5 model API

*I have 20 new listings to check: do I type each one into the app?* No: call the API directly from Python, the same API the Streamlit app uses. This notebook (like the deployment lesson's L6.2) scores a batch of listings and orders in two ways, one call per record and one batch call, and collects the results in a DataFrame.

Runs anywhere with `requests` and `pandas` (Colab included). The service is on Render's free tier: if it has been idle for 15 minutes the first call takes 30 to 60 seconds while it wakes up. The endpoints are documented in `deployment/api/README.md` and at `/docs`.

In [1]:
import os, time
import pandas as pd
import requests

API = os.environ.get("API_URL", "https://team5-olist-api.onrender.com")


def wake(api, limit=180):
    """Ping the health endpoint until the service answers (a sleeping free-tier service takes 30-60 s)."""
    t0 = time.time()
    while time.time() - t0 < limit:
        try:
            r = requests.get(f"{api}/", timeout=60)
            if r.ok:
                return r.json() | {"answered_after_s": round(time.time() - t0)}
        except requests.RequestException:
            pass
        time.sleep(5)
    raise RuntimeError(f"{api} did not answer")


wake(API)

{'status': 'healthy',
 'service': 'Team 5 Olist model API',
 'version': '1.0.0',
 'models_loaded': True,
 'models_built': '2026-10-29',
 'uptime_s': 362,
 'endpoints': ['/model-info',
  '/freight/predict',
  '/freight/batch',
  '/late/predict',
  '/late/batch',
  '/logs/summary',
  '/docs'],
 'answered_after_s': 0}

In [2]:
info = requests.get(f"{API}/model-info", timeout=60).json()
pd.DataFrame({name: {"role": m["role"], "estimator": m["estimator"], "test": m["test"]}
              for name, m in info["models"].items()}).T

,role,estimator,test
freight_forest,fair freight,RandomForestRegressor on log(1 + freight),"{'MAE_rs': 5.398, 'RMSE_rs': 12.032, 'R2_rs': ..."
freight_ridge,second opinion on unusual charges,Ridge on log(1 + freight),"{'MAE_rs': 6.123, 'RMSE_rs': 14.243, 'R2_rs': ..."
late_logit,late-delivery risk ranking,LogisticRegression (no penalty),"{'Lift': 1.998, 'PR_AUC': 0.07, 'ROC_AUC': 0.6..."


## 1. Twenty new listings, one call each

Each listing gives the item (price, weight, package volume, category), the route (seller and customer state, optionally the zip codes) and the freight the seller charged. The API returns the fair freight, a quote range and whether the charge is unusual.

In [3]:
listings = pd.DataFrame([
    ("bed_bath_table", 120.0, 1.50, 9000, "SP", "RJ", "01310", "20040", 35.00),
    ("health_beauty", 45.0, 0.30, 1500, "SP", "BA", None, None, 12.50),
    ("sports_leisure", 89.9, 0.80, 6000, "SP", "SP", "01310", "13087", 9.90),
    ("furniture_decor", 250.0, 8.00, 60000, "PR", "MG", None, None, 48.00),
    ("computers_accessories", 450.0, 2.20, 12000, "SP", "PE", None, None, 39.00),
    ("housewares", 35.0, 0.40, 2400, "MG", "MG", "30140", "30130", 22.00),
    ("watches_gifts", 199.0, 0.25, 1000, "SP", "RS", None, None, 18.00),
    ("garden_tools", 79.0, 3.50, 30000, "SC", "SP", None, None, 41.00),
    ("telephony", 29.9, 0.15, 800, "SP", "AM", None, None, 25.00),
    ("toys", 60.0, 1.20, 8000, "RJ", "RJ", None, None, 15.00),
    ("cool_stuff", 150.0, 2.00, 15000, "SP", "CE", None, None, 95.00),
    ("auto", 90.0, 1.80, 7000, "SP", "GO", None, None, 21.00),
    ("perfumery", 110.0, 0.50, 2000, "SP", "SP", None, None, 8.50),
    ("baby", 70.0, 1.00, 10000, "PR", "SC", None, None, 17.00),
    ("electronics", 39.0, 0.20, 1200, "SP", "MA", None, None, 26.00),
    ("stationery", 25.0, 0.60, 3000, "SP", "RJ", None, None, 16.00),
    ("office_furniture", 380.0, 14.00, 90000, "SP", "SP", None, None, 120.00),
    ("pet_shop", 55.0, 2.50, 9000, "MG", "SP", None, None, 19.00),
    ("luggage_accessories", 230.0, 5.00, 70000, "SP", "PA", None, None, 70.00),
    ("musical_instruments", 900.0, 6.00, 50000, "SP", "DF", None, None, 45.00),
], columns=["category", "price", "weight_kg", "volume_cm3", "seller_state", "customer_state",
            "seller_zip", "customer_zip", "freight_value"])
records = [{k: v for k, v in r.items() if v is not None} for r in listings.to_dict("records")]
len(records)

20

In [4]:
t0 = time.time()
one_by_one = []
print(f"Scoring {len(records)} listings, one call each...")
print("=" * 86)
for i, rec in enumerate(records, 1):
    p = requests.post(f"{API}/freight/predict", json=rec, timeout=90).json()
    one_by_one.append(p)
    v = p["check"]["rules"]["band"]["verdict"]
    print(f"[{i:2}/{len(records)}] {rec['category'][:20]:<20} {rec['seller_state']}->{rec['customer_state']}  "
          f"charged R$ {rec['freight_value']:6.2f}  fair R$ {p['fair_freight']:6.2f}  "
          f"quote {p['quote_low']:4.0f}-{p['quote_high']:<4.0f} {v}")
print("=" * 86)
print(f"Completed {len(records)} calls in {time.time() - t0:.1f} s")

Scoring 20 listings, one call each...
[ 1/20] bed_bath_table       SP->RJ  charged R$  35.00  fair R$  18.34  quote   11-25   normal
[ 2/20] health_beauty        SP->BA  charged R$  12.50  fair R$  17.77  quote   15-21   normal
[ 3/20] sports_leisure       SP->SP  charged R$   9.90  fair R$  10.59  quote    6-15   normal
[ 4/20] furniture_decor      PR->MG  charged R$  48.00  fair R$  30.98  quote   20-42   normal


[ 5/20] computers_accessorie SP->PE  charged R$  39.00  fair R$  29.61  quote   19-40   normal
[ 6/20] housewares           MG->MG  charged R$  22.00  fair R$   8.83  quote    6-12   unusual_both
[ 7/20] watches_gifts        SP->RS  charged R$  18.00  fair R$  15.03  quote    8-22   normal
[ 8/20] garden_tools         SC->SP  charged R$  41.00  fair R$  21.91  quote   18-26   normal
[ 9/20] telephony            SP->AM  charged R$  25.00  fair R$  23.08  quote   20-26   normal


[10/20] toys                 RJ->RJ  charged R$  15.00  fair R$  10.36  quote    6-15   normal
[11/20] cool_stuff           SP->CE  charged R$  95.00  fair R$  30.83  quote   24-38   unusual_both
[12/20] auto                 SP->GO  charged R$  21.00  fair R$  18.82  quote   14-23   normal
[13/20] perfumery            SP->SP  charged R$   8.50  fair R$   9.02  quote    2-16   normal


[14/20] baby                 PR->SC  charged R$  17.00  fair R$  17.14  quote   13-22   normal
[15/20] electronics          SP->MA  charged R$  26.00  fair R$  28.74  quote   26-32   normal
[16/20] stationery           SP->RJ  charged R$  16.00  fair R$  15.55  quote   12-19   normal
[17/20] office_furniture     SP->SP  charged R$ 120.00  fair R$  33.59  quote   23-44   unusual_both


[18/20] pet_shop             MG->SP  charged R$  19.00  fair R$  18.89  quote   15-23   normal
[19/20] luggage_accessories  SP->PA  charged R$  70.00  fair R$  66.20  quote   56-77   normal
[20/20] musical_instruments  SP->DF  charged R$  45.00  fair R$  34.16  quote   17-51   normal
Completed 20 calls in 1.1 s


## 2. The same listings in one batch call

`/freight/batch` takes up to 5,000 records per call, so a file of any size goes in a few calls (`batch_score.py` sends the 22,529 held-out items in chunks of 2,000).

In [5]:
t0 = time.time()
res = requests.post(f"{API}/freight/batch", json={"items": records}, timeout=300).json()
print(f"{res['n']} listings in one call: {time.time() - t0:.2f} s round trip, "
      f"{res['latency_ms']:.0f} ms inside the API; {res['n_with_drift']} with drift warnings")

scored = listings.assign(
    fair_freight=[p["fair_freight"] for p in res["predictions"]],
    ridge=[p["fair_freight_ridge"] for p in res["predictions"]],
    quote=[f"{p['quote_low']:.0f}-{p['quote_high']:.0f}" for p in res["predictions"]],
    distance_km=[p["inputs"]["distance_km"] for p in res["predictions"]],
    overall=[p["check"]["rules"]["overall"]["verdict"] for p in res["predictions"]],
    per_band=[p["check"]["rules"]["band"]["verdict"] for p in res["predictions"]],
    drift=["; ".join(p["drift_warnings"]) for p in res["predictions"]])
assert [p["fair_freight"] for p in one_by_one] == list(scored.fair_freight)   # same answer either way
scored.drop(columns=["seller_zip", "customer_zip"]).round(2)

20 listings in one call: 0.05 s round trip, 35 ms inside the API; 4 with drift warnings


,category,price,weight_kg,volume_cm3,seller_state,customer_state,freight_value,fair_freight,ridge,quote,distance_km,overall,per_band,drift
0,bed_bath_table,120.0,1.50,9000,SP,RJ,35.0,18.34,17.89,11-25,362.91,normal,normal,
1,health_beauty,45.0,0.30,1500,SP,BA,12.5,17.77,17.65,15-21,1430.17,normal,normal,
2,sports_leisure,89.9,0.80,6000,SP,SP,9.9,10.59,11.42,6-15,88.35,normal,normal,
3,furniture_decor,250.0,8.00,60000,PR,MG,48.0,30.98,33.53,20-42,933.20,normal,normal,
4,computers_accessories,450.0,2.20,12000,SP,PE,39.0,29.61,35.31,19-40,2117.87,normal,normal,
5,housewares,35.0,0.40,2400,MG,MG,22.0,8.83,7.43,6-12,0.12,unusual_both,unusual_both,distance_km = 0.12 is below the training mean ...
6,watches_gifts,199.0,0.25,1000,SP,RS,18.0,15.03,14.59,8-22,846.94,normal,normal,
7,garden_tools,79.0,3.50,30000,SC,SP,41.0,21.91,23.19,18-26,477.62,normal,normal,
8,telephony,29.9,0.15,800,SP,AM,25.0,23.08,24.04,20-26,2686.29,normal,normal,
9,toys,60.0,1.20,8000,RJ,RJ,15.0,10.36,8.20,6-15,0.00,normal,normal,distance_km = 0.00 is below the training mean ...


In [6]:
summary = pd.DataFrame({rule: scored[rule].value_counts() for rule in ["overall", "per_band"]}).fillna(0).astype(int)
summary.columns = ["Top 5% overall (evaluated)", "Top 5% within each price band (recommended)"]
summary
# scored.to_csv("scored_listings.csv", index=False)   # hand the list to the pricing team

,Top 5% overall (evaluated),Top 5% within each price band (recommended)
normal,17,17
unusual_both,3,3


## 3. Late-delivery risk for a batch of orders

`/late/batch` ranks orders at checkout: the risk percentile among the test-window orders, whether the order falls in the riskiest 10% (`share`), what drives its risk and a suggested low-cost follow-up.

In [7]:
orders = [
    dict(promised_days=12, price_total=150, freight_total=25, seller_state="SP", customer_state="BA"),
    dict(promised_days=30, price_total=150, freight_total=25, seller_state="SP", customer_state="BA"),
    dict(promised_days=8, price_total=89, freight_total=12, seller_state="SP", customer_state="SP"),
    dict(promised_days=25, price_total=420, freight_total=60, seller_state="PR", customer_state="RJ", n_items=3, n_sellers=2),
    dict(promised_days=15, price_total=60, freight_total=18, seller_state="MG", customer_state="MA"),
    dict(promised_days=45, price_total=230, freight_total=38, seller_state="SP", customer_state="AM"),
]
res = requests.post(f"{API}/late/batch", json={"orders": orders}, params={"share": 0.10}, timeout=120).json()
pd.DataFrame(orders).fillna({"n_items": 1, "n_sellers": 1}).astype({"n_items": int, "n_sellers": int}).assign(
    risk=[p["risk"] for p in res["predictions"]],
    percentile=[p["percentile"] for p in res["predictions"]],
    on_review_list=[p["on_review_list"] for p in res["predictions"]],
    top_driver=[p["top_driver"] for p in res["predictions"]],
    follow_up=[p["suggested_follow_up"] for p in res["predictions"]]).round(3)

,promised_days,price_total,freight_total,seller_state,customer_state,n_items,n_sellers,risk,percentile,on_review_list,top_driver,follow_up
0,12,150,25,SP,BA,1,1,0.485,99.297,True,promised_days,Check carrier tracking
1,30,150,25,SP,BA,1,1,0.150,81.216,False,customer_state,Check carrier tracking
2,8,89,12,SP,SP,1,1,0.028,15.723,False,promised_days,Update the delivery estimate
3,25,420,60,PR,RJ,3,2,0.030,17.090,False,customer_state,Check carrier tracking
4,15,60,18,MG,MA,1,1,0.287,95.610,True,customer_state,Check carrier tracking
5,45,230,38,SP,AM,1,1,0.045,28.619,False,distance_km,Check carrier tracking


## 4. What the service logged

Every prediction above went into the API's inference log; `/logs/summary` aggregates it for monitoring. On the free tier the log resets when the service sleeps.

In [8]:
s = requests.get(f"{API}/logs/summary", timeout=60).json()
{k: s[k] for k in ["requests", "predictions", "drift", "latency_ms"]}

{'requests': {'total': 85,
  'by_endpoint': {'/freight/batch': 18,
   '/late/batch': 18,
   '/freight/predict': 46,
   '/late/predict': 3},
  'rejected_422': 8},
 'predictions': {'total': 42683, 'freight': 23025, 'late': 19658},
 'drift': {'predictions_with_drift': 1849,
  'rate': 0.0433,
  'by_input': {'distance_km': 441,
   'freight_ratio': 355,
   'promised_days': 340,
   'freight_total': 334,
   'price': 181,
   'weight_kg': 175,
   'price_total': 118,
   'chargeable_kg': 51,
   'category': 2}},
 'latency_ms': {'requests': 85, 'p50': 38.6, 'p95': 410.5, 'max': 556.6}}